In [5]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [42]:
# All imports in one place
import pandas as pd
import joblib

# For fetching stock data
import yfinance as yf

# Required for Model development and Training
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

print("All libraries loaded successfully.")

All libraries loaded successfully.


In [6]:
import pandas as pd
import csv
# Load the cleaned Apple sentiment dataset
path = "/content/drive/MyDrive/COMP331/AAPL_Cleaned_Sentiment.csv"


In [7]:
df = pd.read_csv(path, engine="python")


In [34]:
df.head()


,Unnamed: 0,id,body,created_at,user,source,symbols,mentioned_users,entities,likes,links,conversation,reshare_message,reshares,owned_symbols,structurable,username,sentiment_clean,date,is_low_quality
0,521,323006332,$AMZN &amp; i’m out. ez call. expecting an aap...,2021-04-29T20:19:27Z,"{'id': 2028386, 'username': 'lego3072', 'name'...","{'id': 1149, 'title': 'StockTwits for iOS', 'u...","[{'id': 864, 'symbol': 'AMZN', 'title': 'Amazo...",[],{'sentiment': None},NaN,NaN,NaN,"{'reshared_count': 1, 'reshared_deleted': Fals...",NaN,NaN,NaN,lego3072,Neutral/None,2021-04-29,False
1,803,322997427,"$AMZN aapl , amd. Perfect example for tomorrow",2021-04-29T20:07:41Z,"{'id': 1124284, 'username': 'rileyvang', 'name...","{'id': 2095, 'title': 'StockTwits For Android ...","[{'id': 864, 'symbol': 'AMZN', 'title': 'Amazo...",[],{'sentiment': {'basic': 'Bearish'}},NaN,NaN,NaN,NaN,NaN,NaN,NaN,rileyvang,Bearish,2021-04-29,False
2,1118,322992237,"$AMZN Looking at what has happened with GOOG, ...",2021-04-29T20:01:35Z,"{'id': 4709541, 'username': 'Modest_Investor_1...","{'id': 2269, 'title': 'StockTwits Web', 'url':...","[{'id': 864, 'symbol': 'AMZN', 'title': 'Amazo...",[],{'sentiment': {'basic': 'Bullish'}},NaN,NaN,NaN,NaN,NaN,NaN,NaN,Modest_Investor_18,Bullish,2021-04-29,False
3,1280,322980176,$AMZN who thinks this will follow GOOG after E...,2021-04-29T19:44:19Z,"{'id': 1577782, 'username': 'InvestIQ', 'name'...","{'id': 1149, 'title': 'StockTwits for iOS', 'u...","[{'id': 864, 'symbol': 'AMZN', 'title': 'Amazo...",[],{'sentiment': {'basic': 'Bullish'}},"{'total': 2, 'user_ids': [1452854, 4319329]}",NaN,NaN,NaN,NaN,NaN,NaN,InvestIQ,Bullish,2021-04-29,False
4,1423,322956747,$AMZN Where AMZN goes one we go all.\n\nNot lo...,2021-04-29T19:04:45Z,"{'id': 3928814, 'username': 'LegionTrading', '...","{'id': 2269, 'title': 'StockTwits Web', 'url':...","[{'id': 864, 'symbol': 'AMZN', 'title': 'Amazo...",[],{'sentiment': {'basic': 'Bullish'}},"{'total': 3, 'user_ids': [4163201, 3579849, 52...",NaN,NaN,NaN,NaN,NaN,NaN,LegionTrading,Bullish,2021-04-29,False


In [ ]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 771914 entries, 0 to 771913
Data columns (total 20 columns):
 #   Column           Non-Null Count   Dtype 
---  ------           --------------   ----- 
 0   Unnamed: 0       771914 non-null  int64 
 1   id               771914 non-null  int64 
 2   body             771914 non-null  object
 3   created_at       771914 non-null  object
 4   user             771914 non-null  object
 5   source           771914 non-null  object
 6   symbols          771914 non-null  object
 7   mentioned_users  771914 non-null  object
 8   entities         771914 non-null  object
 9   likes            491352 non-null  object
 10  links            72206 non-null   object
 11  conversation     256466 non-null  object
 12  reshare_message  31366 non-null   object
 13  reshares         29213 non-null   object
 14  owned_symbols    24069 non-null   object
 15  structurable     1 non-null       object
 16  username         771914 non-null  object
 17  sentiment_

In [9]:
# Convert the 'date' column to datetime
df['date'] = pd.to_datetime(df['date']).dt.date

print(df.shape)
print(df.columns)

(771914, 20)
Index(['Unnamed: 0', 'id', 'body', 'created_at', 'user', 'source', 'symbols',
       'mentioned_users', 'entities', 'likes', 'links', 'conversation',
       'reshare_message', 'reshares', 'owned_symbols', 'structurable',
       'username', 'sentiment_clean', 'date', 'is_low_quality'],
      dtype='object')


In [20]:
# Download Apple stock prices (2020-2023)
apple_price = yf.download("AAPL", start="2020-01-01", end="2023-01-01")
apple_price = apple_price.reset_index()

# Flatten any multi-level columns
apple_price.columns = [col if isinstance(col, str) else col[0] for col in apple_price.columns]

# Make sure 'Date' is present
print(apple_price.columns)
apple_price['Date'] = pd.to_datetime(apple_price['Date']).dt.date

print(f"Stock prices loaded: {len(apple_price)} trading days")
apple_price.head()

/tmp/ipykernel_1739/3716530853.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  apple_price = yf.download("AAPL", start="2020-01-01", end="2023-01-01")
[*********************100%***********************]  1 of 1 completed

Index(['Date', 'Close', 'High', 'Low', 'Open', 'Volume'], dtype='object')
Stock prices loaded: 756 trading days


,Date,Close,High,Low,Open,Volume
0,2020-01-02,72.400528,72.460791,71.156689,71.409793,135480400
1,2020-01-03,71.696648,72.455966,71.472469,71.629153,146322800
2,2020-01-06,72.267929,72.306499,70.568503,70.819201,118387200
3,2020-01-07,71.928047,72.533087,71.708687,72.277571,108872000
4,2020-01-08,73.085091,73.386408,71.631537,71.631537,132079200


In [11]:
# 1. Convert BOTH date columns to datetime.date so they match perfectly (once again for safety)
apple_price['Date'] = pd.to_datetime(apple_price['Date']).dt.date
df['date'] = pd.to_datetime(df['date']).dt.date

# 2. Compute next-day price change using unique trading days
daily_prices = apple_price[['Date', 'Close']].drop_duplicates().sort_values('Date')
daily_prices['Next_Close'] = daily_prices['Close'].shift(-1)
daily_prices['Price_Change'] = daily_prices['Next_Close'] - daily_prices['Close']

# 3. Merge sentiment posts with daily prices
merged_df = pd.merge(
    df,
    daily_prices[['Date', 'Close', 'Price_Change']],
    left_on='date',
    right_on='Date',
    how='left'
)

# Fill weekend/holiday gaps — pairs weekend posts with next Monday's price action
merged_df[['Close', 'Price_Change']] = merged_df[['Close', 'Price_Change']].bfill()

print(f"Merged dataset: {merged_df.shape[0]} rows")
# Check the first few rows to verify
merged_df[['date', 'body', 'sentiment_clean', 'Close', 'Price_Change']].head()


Merged dataset: 771914 rows


,date,body,sentiment_clean,Close,Price_Change
0,2021-04-29,$AMZN &amp; i’m out. ez call. expecting an aap...,Neutral/None,130.008896,-1.967422
1,2021-04-29,"$AMZN aapl , amd. Perfect example for tomorrow",Bearish,130.008896,-1.967422
2,2021-04-29,"$AMZN Looking at what has happened with GOOG, ...",Bullish,130.008896,-1.967422
3,2021-04-29,$AMZN who thinks this will follow GOOG after E...,Bullish,130.008896,-1.967422
4,2021-04-29,$AMZN Where AMZN goes one we go all.\n\nNot lo...,Bullish,130.008896,-1.967422


In [24]:
print("Save merged dataset to Drive (raw baseline with stock prices)")
print("*" * 100 + "\nThese lines the commented out as we do not need to save it again")
  # merged_path = "/content/drive/MyDrive/COMP331/AAPL_Merged_Sentiment_Stock.csv"
  # merged_df.to_csv(merged_path, index=False)
  # print(f"Merged dataset saved: {len(merged_df):,} rows → {merged_path}")

Save merged dataset to Drive (raw baseline with stock prices)
****************************************************************************************************
These lines the commented out as we do not need to save it again


In [ ]:
# Check all unique values and how many times they appear
print(merged_df['sentiment_clean'].value_counts(dropna=False))

sentiment_clean
Neutral/None    383369
Bullish         295581
Bearish          92964
Name: count, dtype: int64


In [26]:
# Labeled posts (have sentiment)
labeled_df = merged_df[merged_df['sentiment_clean'].isin(['Bullish', 'Bearish'])]

# Unlabeled posts (missing sentiment)
unlabeled_df = merged_df[merged_df['sentiment_clean'] == 'Neutral/None']

print(f"Labeled rows for training: {len(labeled_df)}")
print(f"Unlabeled rows to predict: {len(unlabeled_df)}")

Labeled rows for training: 388545
Unlabeled rows to predict: 383369


In [27]:
### from sklearn.feature_extraction.text import TfidfVectorizer

# Vectorize post text using TF-IDF
# Fit on ALL posts so the vocabulary is consistent across labeled and unlabeled
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
print("Vectorizing text... this may take a minute.")
vectorizer.fit(merged_df['body'].fillna(''))

X_labeled = vectorizer.transform(labeled_df['body'].fillna(''))
y_labeled = labeled_df['sentiment_clean']
X_unlabeled = vectorizer.transform(unlabeled_df['body'].fillna(''))

print(f"TF-IDF matrix shape (labeled): {X_labeled.shape}")

Vectorizing text... this may take a minute.
TF-IDF matrix shape (labeled): (388545, 5000)


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

# Train Decision Tree classifier and evaluate on held-out test set

# Split for evaluation
X_train, X_test, y_train, y_test = train_test_split(
    X_labeled, y_labeled, test_size=0.2, random_state=42
)

print("Shapes:", X_train.shape, y_train.shape)

# Initialize and train
dt_model = DecisionTreeClassifier(class_weight='balanced', random_state=42)
dt_model.fit(X_train, y_train)

# Evaluate
y_pred = dt_model.predict(X_test)

print("=== Decision Tree Model Performance ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print()
print(classification_report(y_test, y_pred))

Shapes: (310836, 5000) (310836,)
Accuracy: 0.742616685326024
              precision    recall  f1-score   support

     Bearish       0.47      0.54      0.50     18725
     Bullish       0.85      0.81      0.83     58984

    accuracy                           0.74     77709
   macro avg       0.66      0.67      0.66     77709
weighted avg       0.76      0.74      0.75     77709



In [46]:
import joblib

# Load the saved model and vectorizer from Drive
dt_model = joblib.load('/content/drive/MyDrive/sentiment_decision_tree.joblib')
vectorizer = joblib.load('/content/drive/MyDrive/sentiment_tfidf_vectorizer.joblib')

# Test it on any new text instantly
new_posts = [
    "Apple earnings looked amazing, buying more shares!",
    "iPhone sales are terrible, selling everything.",
    "AAPL just hit a new all time high"
]

new_vectorized = vectorizer.transform(new_posts)
predictions = dt_model.predict(new_vectorized)

for post, sentiment in zip(new_posts, predictions):
    print(f"{sentiment:8} → {post}")

Bearish  → Apple earnings looked amazing, buying more shares!
Bullish  → iPhone sales are terrible, selling everything.
Bullish  → AAPL just hit a new all time high


In [48]:
# Predict sentiment for all missing unlabeled posts
print(f"Predicting sentiment for {len(unlabeled_df):,} unlabeled rows...")
predicted_sentiments = dt_model.predict(X_unlabeled)

# Assign the predictions back to the unlabeled dataframe
unlabeled_df_predicted = unlabeled_df.copy()
unlabeled_df_predicted['sentiment_clean'] = predicted_sentiments

# Combine originally labeled + newly predicted into one full dataset
final_cleaned_df = pd.concat([labeled_df, unlabeled_df_predicted])

print(f"\nFinal completed dataset: {len(final_cleaned_df)} rows")
print("\nNew label distribution:")
print(final_cleaned_df['sentiment_clean'].value_counts())

Predicting sentiment for 383,369 unlabeled rows...

Final completed dataset: 771914 rows

New label distribution:
sentiment_clean
Bullish    564453
Bearish    207461
Name: count, dtype: int64


In [49]:

# Evaluate Predictive Accuracy

print("--- Calculating Daily Predictive Accuracy ---")

# Group the data by date.
# We find the most common sentiment (mode) for each day and grab that day's price change.
daily_summary = final_cleaned_df.groupby('date').agg(
    majority_sentiment=('sentiment_clean', lambda x: x.mode()[0]),
    price_change=('Price_Change', 'first')
).dropna() # Drop any trailing weekends that don't have a future price change

# Define the logic: Did the crowd's sentiment correctly guess the market direction?
def check_accuracy(row):
    if row['majority_sentiment'] == 'Bullish' and row['price_change'] > 0:
        return 1  # Correct prediction (Bullish and stock went up)
    elif row['majority_sentiment'] == 'Bearish' and row['price_change'] < 0:
        return 1  # Correct prediction (Bearish and stock went down)
    else:
        return 0  # Incorrect prediction

# Apply the logic to our daily summary
daily_summary['correct_prediction'] = daily_summary.apply(check_accuracy, axis=1)

# Calculate and print the final metric
total_days = len(daily_summary)
correct_days = daily_summary['correct_prediction'].sum()
accuracy = (correct_days / total_days) * 100

print(f"Total Trading Days Analyzed: {total_days}")
print(f"Correctly Predicted Days: {correct_days}")
print(f"Cleaned Data Predictive Accuracy: {accuracy:.2f}%")

--- Calculating Daily Predictive Accuracy ---
Total Trading Days Analyzed: 2242
Correctly Predicted Days: 1268
Cleaned Data Predictive Accuracy: 56.56%


In [33]:
print("For saving the Model and Vectorizer to Google Drive")
print("*" * 50 + "\nThese lines the commented out as we do not need to save it again")
# # 1. Define the file paths in your Google Drive
# model_path = '/content/drive/MyDrive/sentiment_decision_tree.joblib'
# vectorizer_path = '/content/drive/MyDrive/sentiment_tfidf_vectorizer.joblib'

# # 2. Save the model and the vectorizer
# joblib.dump(dt_model, model_path)
# joblib.dump(vectorizer, vectorizer_path)

# print("Model and vectorizer successfully saved to Google Drive!")

For saving the Model and Vectorizer to Google Drive
**************************************************
These lines the commented out as we do not need to save it again


In [ ]:
# import joblib

# # Load the saved files
# loaded_model = joblib.load('/content/drive/MyDrive/sentiment_decision_tree.joblib')
# loaded_vectorizer = joblib.load('/content/drive/MyDrive/sentiment_tfidf_vectorizer.joblib')

# # Now you can instantly predict new text!
# new_post = ["Apple earnings looked terrible today, selling all my shares."]
# new_post_vectorized = loaded_vectorizer.transform(new_post)
# prediction = loaded_model.predict(new_post_vectorized)

# print(prediction) # Output will likely be ['Bearish']